# Парсинг недостающих данных

#### Выполнил X

После предыдущего этапа в датасете остались пропуски, которые необходимо заполнить. Обратимся с этой целью к парсингу данных с площадок-агреггаторов:

<ul>
    <li>IMDB</li>
    <li>Box office Mojo</li>
    <li>The numbers</li>
</ul>

In [ ]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import re
from IPython.display import clear_output

# Проверка пропусков

Выполним загрузку датасета из прошлого этапа:

In [173]:
df = pd.read_csv("netflix_enriched_2.csv")
df.shape

(498, 23)

In [174]:
df.head(5)

,title,rating,rating_level,release_year,rating_score_nfx,rating_size_nfx,title_lower,imdb_id,budget,revenue,...,vote_average_imdb,vote_count_imdb,genres,genres_list,genres_consolidated,is_action,is_comedy,is_drama,is_family,is_other
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80,white chicks,tt0381707,37000000.0,113086475.0,...,5.9,193002.0,"Comedy, Crime","['Comedy', 'Crime']","['Action', 'Comedy']",1,1,0,0,0
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,79.0,82,lucky number slevin,tt0425210,27000000.0,56308881.0,...,7.7,338824.0,"Drama, Thriller, Crime, Mystery","['Drama', 'Thriller', 'Crime', 'Mystery']","['Action', 'Drama', 'Other']",1,0,1,0,1
2,Death Note,TV-14,Parents strongly cautioned. May be unsuitable ...,2006,77.0,80,death note,tt0758742,20000000.0,29667169.0,...,7.5,33333.0,"Fantasy, Mystery, Thriller","['Fantasy', 'Mystery', 'Thriller']","['Action', 'Other']",1,0,0,0,1
3,The Hunter,R,language and brief violence,2011,79.0,82,the hunter,tt1703148,NaN,176669.0,...,6.7,42712.0,"Drama, Thriller, Adventure","['Drama', 'Thriller', 'Adventure']","['Action', 'Drama']",1,0,1,0,0
4,The Do-Over,TV-MA,For mature audiences. May not be suitable for...,2016,84.0,80,the do-over,tt4769836,NaN,NaN,...,5.7,57047.0,"Action, Adventure, Comedy","['Action', 'Adventure', 'Comedy']","['Action', 'Comedy']",1,1,0,0,0


Проверим число строк с пропусками, которые необходимо заполнить:

In [175]:
df['imdb_votecount_new'] = np.nan

columns_data_missing = [
    "runtime",
    "vote_average_imdb", 
    'genres',
    "imdb_votecount_new"
]

df.isna().sum()

title                    0
rating                   0
rating_level             0
release_year             0
rating_score_nfx         0
rating_size_nfx          0
title_lower              0
imdb_id                175
budget                 390
revenue                395
runtime                184
vote_average_tmdb      281
vote_count_tmdb        281
vote_average_imdb      283
vote_count_imdb        283
genres                 179
genres_list              0
genres_consolidated      0
is_action                0
is_comedy                0
is_drama                 0
is_family                0
is_other                 0
imdb_votecount_new     498
dtype: int64

Запустим selenium:

In [176]:
driver = webdriver.Chrome()
driver.maximize_window()

# Парсинг IMDB

Спарсим данные с агрегатора IMDB.

In [177]:
df_imdb = df.copy()

In [178]:
imdb_counter = 1

def get_movie_page_HTML(row: pd.Series) -> str:
    url = f"https://www.imdb.com/find/?q={row["title"]}"
    driver.get(url)

    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, '[data-testid="find-results-section-title"]')
        )
    )

    driver.find_element(By.CLASS_NAME, 'ipc-lockup-overlay__screen').click()

    driver.execute_script("""
        window.scrollTo(
            0,
            document.body.scrollHeight * 0.55
        );
    """)

    if pd.isna(row['genres']):
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, '[data-testid="storyline-genres"]')
                )
            )
        except: pass

    print(driver.current_url)
    return driver.page_source
    
def print_stat(row: pd.Series):
        global imdb_counter
        print("*" * 20)
        print(f"Итерация {imdb_counter}/{df_imdb.shape[0]}. Проект {row['title']}")
        print("*" * 20)
        imdb_counter += 1
        clear_output(wait=True)

def get_imdb_rating(soup: BeautifulSoup) -> float | None:
    rating_span = soup.select_one("span.sc-a30a09c4-1.leFYws")
    if rating_span: return float(rating_span.text.strip())
    else: return None

def get_imdb_runtime(soup: BeautifulSoup) -> str | None:
    ul_runtime = soup.find('ul', class_="ipc-inline-list ipc-inline-list--show-dividers sc-b41e510f-3 ggypaO baseAlt baseAlt")

    if ul_runtime: 
        ul_runtime = ul_runtime.find_all("li")

        for li in ul_runtime:
            text = li.get_text(strip=True)
            if re.search(r"^(?:\d+h\s*\d+m|\d+h|\d+m)$", text):
                return text.strip()

    else: return None

def get_imdb_votecount(soup: BeautifulSoup) -> str | None:
    vote_count = soup.find("span", class_="vote-count")

    if vote_count: return vote_count.text
    else: return None

def get_imdb_genres(soup: BeautifulSoup) -> str | None:
    genres_div = soup.select_one('[data-testid="storyline-genres"]')
    result = []
    if genres_div:
        genres_li = genres_div.find_all("li", class_="ipc-inline-list__item")

        for genre in genres_li:
            result.append(genre.text.strip())
        
        return ", ".join(result)

    return None

imdb_func_exec = {
    "runtime": get_imdb_runtime, 
    "imdb_votecount_new": get_imdb_votecount, 
    "vote_average_imdb": get_imdb_rating, 
    'genres': get_imdb_genres,
}


In [179]:
def parse_imdb(row: pd.Series):

    row_copy = row
    try:
        html = get_movie_page_HTML(row_copy)
        soup = BeautifulSoup(html, "lxml")
        for column in columns_data_missing:
            try:
                if pd.isna(row_copy[column]):
                    fill_value = imdb_func_exec[column](soup)
                    if fill_value:
                        row_copy[column] = fill_value
                    print(f"{column}: {'не найдено' if not fill_value else fill_value}")
            except: 
                print(f"{column}: ошибка")
                continue
    except:
        print("Ошибка")
        return row
    finally:
        print_stat(row)
    
    return row_copy

In [180]:
df_imdb = df_imdb.apply(parse_imdb, axis=1)

https://www.imdb.com/title/tt6780296/?ref_=fn_i_1
vote_average_imdb: 8.2
imdb_votecount_new: 94
********************
Итерация 498/498. Проект Beary Tales
********************


In [181]:
df_imdb.to_csv('result.csv')

In [182]:
df_result = pd.read_csv('result.csv')

df_result[columns_data_missing].isna().sum()

runtime               18
vote_average_imdb     18
genres                30
imdb_votecount_new    33
dtype: int64

In [ ]:
driver.quit()